# Website Chatbot using Retrieval-Augmented Generation (RAG)

# Problem Statement

Modern websites often contain hundreds of pages of documentation, product information, blogs, FAQs, pricing details, and support content. Finding a specific piece of information typically requires users to manually navigate through multiple pages or rely on keyword-based search, which may not always provide the desired results.

Traditional Large Language Models (LLMs) cannot answer questions about the content of a specific website unless that information was part of their training data. Even when the information exists publicly, an LLM may generate incomplete or inaccurate responses because it does not have direct access to the latest website contnsufficient.

To address these challenges, we use **Retrieval-Augmented Generaand reliability.

The objective of this project is to build an intelligent Website Chatbot capable of:

- Processing the content of a website.
- Understanding user questions in natural language.
- Retrieving the most relevant information using semantic search.
- Generating accurate, context-aware answers.
- Displaying the source content used to produce the response.


## Project Overview

This project demonstrates how to build an AI-powered Website Chatbot using the Retrieval-Augmented Generation (RAG) architecture.

Instead of manually reading an entire website, users can simply provide the website URL and ask questions in natural language. The chatbot retrieves the most relevant information from the website and uses a Large Language Model (LLM) to generate accurate, context-aware responses.

This notebook is designed as a complete learning resource. Every stage of the RAG pipeline is explained conceptually before implementation, making it suitable for beginners while following production-quality development pra into a modular Streamlit application suitable for deployment.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
def load_website(website_url):
    loader = WebBaseLoader(website_url)
    return loader.load()

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
def split_text(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    return text_splitter.split_documents(documents)

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
from langchain_chroma import Chroma

In [14]:
def vector(chunks):
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model
    )
    return vector_db

In [15]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
from langchain_core.prompts import ChatPromptTemplate

In [18]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant answering questions about a website.

Carefully analyze the entire context before answering.

If multiple retrieved passages describe different parts of the website, combine them into one coherent answer.

Do not make assumptions outside the provided context.

If the answer is unavailable, reply:

"I couldn't find the answer on the provided website."

Context:
{context}

Question:
{question}

Answer:"""
)

In [20]:
from dotenv import load_dotenv
import os

In [21]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [23]:
from langchain_groq import ChatGroq

In [24]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.4
)

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [27]:
def website_process(website_url):
    documents = load_website(website_url)
    chunks = split_text(documents)
    vector_db = vector(chunks)

    return vector_db.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 8,
            "fetch_k": 20,
            "lambda_mult": 0.5
        }
    )

In [47]:
retriever = website_process("https://www.linkedin.com/feed/")

In [49]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt | llm | StrOutputParser()
)

In [54]:
ans = rag_chain.invoke("What is this website for?")

In [56]:
print(ans)

This website appears to be for LinkedIn, a professional networking platform, where users can sign in to their accounts or join the platform if they are new to it. The website allows users to log in using their email, phone, or Apple account, and provides access to various features and services, although the specific features are not detailed in the provided context.
